In [ ]:
from utils import *
from plotly.subplots import make_subplots
from tqdm.auto import tqdm
import json
from loaders import *

In [ ]:
res_df, scalars = adaptive_ratio_v1_0_loader()
res_b_df, scalars_b = baseline_loader()

In [ ]:
envs = res_df["env"].unique()

fig = make_subplots(
    rows=1,
    cols=len(envs),
    column_titles=[*envs],
)


def get_avg_df(df):
    g = df.groupby("tag")
    avg_df = pd.DataFrame.from_records(
        {
            "score_mean": g["score"].mean(),
            "score_std": g["score"].std(),
        }
    )
    avg_df = avg_df.reset_index()
    avg_df = avg_df.sort_values(by="tag")
    return avg_df


for col, env in enumerate(envs, 1):
    df1 = res_df[res_df["env"] == env].copy()
    df1["tag"] = "Ada"
    df2 = res_b_df[res_b_df["env"] == env].copy()
    df2["tag"] = df2["ratio"]
    df = pd.concat([df1, df2])
    key = lambda x: 0 if x == "Ada" else x
    df = df.sort_values("tag", key=lambda s: s.apply(key))

    fig.add_trace(
        go.Box(x=df["tag"], y=df["score"], showlegend=False, sizemode="sd"),
        row=1,
        col=col,
    )

    fig.update_xaxes(type="category", row=1, col=col)

fig.update_layout(width=1000, height=400)
fig.write_image("../tex/assets/adaptive_v1_0.pdf")
fig

In [ ]:
envs = res_df["env"].unique()

fig = make_subplots(
    rows=1,
    cols=len(envs),
    column_titles=[*envs],
)


def add_val_losses(df, scalars, key="val/wm_loss"):
    values = []
    for _, row in df.iterrows():
        test_df = scalars.read(row["path"])
        val_losses = test_df[test_df["tag"] == key]
        val_losses = val_losses["value"].to_numpy()
        values.append(val_losses[-1])
    df["val_loss"] = values


val_ranges = {
    "Assault": [1.5, 5.0],
    "CrazyClimber": [0.5, 1.5],
    "MsPacman": [1.2, 2.5],
}


for col, env in enumerate(envs, 1):
    # Final scores
    df1 = res_df[res_df["env"] == env].copy()
    df1["tag"] = "Ada"
    add_val_losses(df1, scalars)
    df2 = res_b_df[res_b_df["env"] == env].copy()
    df2["tag"] = df2["ratio"]
    add_val_losses(df2, scalars_b)
    df = pd.concat([df1, df2])
    key = lambda x: 0 if x == "Ada" else x
    df = df.sort_values("tag", key=lambda s: s.apply(key))

    fig.add_trace(
        go.Box(x=df["tag"], y=df["val_loss"], showlegend=False, sizemode="sd"),
        row=1,
        col=col,
    )

    fig.update_xaxes(type="category", row=1, col=col)


fig.update_layout(width=1000, height=400)
fig.write_image("../tex/assets/adaptive_v1_0.val_loss.pdf")
fig

In [ ]:
envs = res_df["env"].unique()

fig = make_subplots(
    rows=2,
    row_titles=["Val loss", "Update ratio"],
    cols=len(envs),
    column_titles=[*envs],
)

val_ranges = {
    "Assault": [1.5, 5.0],
    "CrazyClimber": [0.5, 1.5],
    "MsPacman": [1.4, 2.5],
}


color_iter = make_color_iter()
colors = [next(color_iter) for _ in range(5)]


# Loss curves
for col, env in enumerate(envs, 1):
    df1 = res_df[res_df["env"] == env].copy()
    dfs = []
    for _, row in df1.iterrows():
        test_df = scalars.read(row["path"])
        test_df = test_df[test_df["tag"] == "ada/val_loss"].copy()
        test_df["seed"] = row["seed"]
        dfs.append(test_df)
    df = pd.concat(dfs)

    for seed in range(2):
        seed_df = df[df["seed"] == seed]
        fig.add_trace(
            go.Scatter(
                x=seed_df["step"],
                y=seed_df["value"],
                legendgroup=f"Seed = {seed}",
                name=f"Seed = {seed}",
                line=dict(color=colors[seed]),
                showlegend=(col == 1),
            ),
            row=1,
            col=col,
        )

    axis = col
    yaxis_name = f"yaxis{axis if axis > 1 else ''}"
    fig.update_layout(**{yaxis_name: dict(range=val_ranges[env])})


# Loss curves
for col, env in enumerate(envs, 1):
    df1 = res_df[res_df["env"] == env].copy()
    dfs = []
    for _, row in df1.iterrows():
        if row["seed"] >= 2:
            continue
        test_df = scalars.read(row["path"])
        test_df = test_df[test_df["tag"] == "ada/wm_ratio"].copy()
        test_df["seed"] = row["seed"]
        dfs.append(test_df)
    df = pd.concat(dfs)

    for seed in range(2):
        seed_df = df[df["seed"] == seed]
        fig.add_trace(
            go.Scatter(
                x=seed_df["step"],
                y=seed_df["value"],
                legendgroup=f"Seed = {seed}",
                name=f"Seed = {seed}",
                line=dict(color=colors[seed]),
                showlegend=False,
            ),
            row=2,
            col=col,
        )

    axis = 3 + col
    yaxis_name = f"yaxis{axis if axis > 1 else ''}"
    fig.update_layout(**{yaxis_name: dict(type="log")})

fig.update_layout(width=1000, height=500)
fig.write_image("../tex/assets/adaptive_v1_0.curves.pdf")
fig